# 🐟 MINA — Phase 1: Fish Gate Classifier

This notebook trains and exports the **MobileNetV3-Small fish/no-fish gate classifier** that will act as a pre-filter before the YOLOv8 disease detection model.

## What this notebook does

1. Clones the repo and checks out the `fish-or-notfish` branch
2. Installs dependencies
3. Configures Kaggle credentials
4. Downloads and organises the training dataset
5. Trains MobileNetV3-Small with two-phase transfer learning
6. Exports the model to ONNX (`fish_gate.onnx`)
7. Evaluates the ONNX model against acceptance criteria
8. Downloads the final `.onnx` file to your machine

## Runtime requirement

> ⚠️ **Make sure you are on a GPU runtime before running this notebook.**
> 
> Go to **Runtime → Change Runtime Type → Hardware Accelerator → T4 GPU**

## Estimated time

| Step | Time |
|---|---|
| Setup & install | ~3–5 min |
| Dataset download | ~3–5 min |
| Training (20 epochs) | ~15–25 min on T4 |
| Export + evaluate | ~2 min |
| **Total** | **~25–40 min** |

---
## Cell 1 — Clone the Repository

Clones `fishcareyolo` from GitHub, checks out the **`fish-or-notfish`** branch (where all the gate classifier code lives), and navigates into `model/`.

**Option A** — Clone from GitHub (default below).

**Option B** — If you prefer to use Google Drive (e.g. you pushed your local changes to Drive instead of GitHub), comment out Option A and uncomment Option B.

In [ ]:
# ── Option A: Clone from GitHub ───────────────────────────────────────────────
# Replace the URL with your actual repo if it's in a private fork.
!git clone https://github.com/MGuruNikhil/fishcareyolo.git
%cd fishcareyolo

# Checkout the branch that contains all the gate classifier code.
# (main does not have this code yet — it lives on fish-or-notfish)
!git checkout fish-or-notfish

%cd model

# ── Option B: Mount Google Drive ──────────────────────────────────────────────
# Use this if you prefer to work from Drive (e.g. after making local changes).
# Uncomment the 4 lines below and comment out Option A above.
#
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/fishcareyolo
# !git checkout fish-or-notfish  # ensure correct branch even from Drive
# %cd model

# Confirm working directory and active branch
import os
print("Working directory:", os.getcwd())
print("Active branch:")
!git branch --show-current
print("Contents:", os.listdir("."))

---
## Cell 2 — Install `uv` and Sync Dependencies

`uv` is the project's package manager (fast drop-in for pip). It reads `pyproject.toml` and installs all declared dependencies into a local virtual environment.

New dependencies added for Phase 1 (already in `pyproject.toml` on `fish-or-notfish`):
- `kaggle>=1.6.0` — Kaggle CLI for dataset download
- `onnxruntime>=1.17.0` — CPU-based ONNX verification after export

> This step typically takes 2–4 minutes on Colab due to `torch`, `tensorflow`, and `ultralytics`.

In [ ]:
# Install uv (fast Python package manager used by this project)
!pip install uv --quiet

# Sync all dependencies declared in pyproject.toml
!uv sync --quiet

# Confirm key packages are available
!uv run python -c "import torch, torchvision, onnxruntime, kaggle; print('All dependencies OK')"

---
## Cell 3 — Configure Kaggle Credentials

The dataset download step (`gate-download`) uses the Kaggle API CLI, which needs your credentials.

### How to get your `kaggle.json`

1. Go to [kaggle.com/settings](https://www.kaggle.com/settings)
2. Scroll to the **API** section
3. Click **Create New Token** — this downloads `kaggle.json`
4. **Upload it to this Colab session** using the Files panel on the left sidebar:  
   Left sidebar → 📁 Files icon → Upload → select `kaggle.json`

> ⚠️ Upload `kaggle.json` BEFORE running this cell. It must be at `/content/kaggle.json`.

In [ ]:
import os
import shutil

# Move kaggle.json to the expected location (~/.kaggle/kaggle.json)
kaggle_src = "/content/kaggle.json"
kaggle_dst_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dst_dir, exist_ok=True)

if not os.path.exists(kaggle_src):
    raise FileNotFoundError(
        "kaggle.json not found at /content/kaggle.json.\n"
        "Please upload it via the Colab Files panel (left sidebar → Files → Upload)."
    )

shutil.copy(kaggle_src, os.path.join(kaggle_dst_dir, "kaggle.json"))
os.chmod(os.path.join(kaggle_dst_dir, "kaggle.json"), 0o600)

# Verify by listing available Kaggle datasets (should not error)
!uv run kaggle datasets list --max-size 1 --quiet
print("Kaggle credentials configured successfully.")

---
## Cell 4 — Download and Organise the Dataset

Runs `gate-download` which does the following automatically:

1. Downloads **fish images** from Kaggle: [`crowww/a-large-scale-fish-dataset`](https://www.kaggle.com/datasets/crowww/a-large-scale-fish-dataset) (~9 000 images, 9 species)
2. Downloads **no-fish images** from Kaggle: [`puneet6060/intel-image-classification`](https://www.kaggle.com/datasets/puneet6060/intel-image-classification) (natural scenes: buildings, forests, glaciers, mountains, seas, streets)
3. Randomly samples **1 000 images per class** (seed=42 for reproducibility)
4. Splits into **train (800) / val (200)** per class
5. Builds the `gate_data/` ImageFolder directory structure
6. Deletes the raw download directories to save disk space

Expected output at the end:
```
  train/fish: 800 images
  train/no_fish: 800 images
  val/fish: 200 images
  val/no_fish: 200 images
```

> This step takes ~3–5 minutes depending on Kaggle download speed.

In [ ]:
# Download both Kaggle datasets and build gate_data/ directory structure
# --keep-raw is omitted so raw downloads are deleted automatically after build
!uv run gate-download

---
## Cell 5 — Verify Dataset Structure

Quick sanity check: confirms all four class splits exist and have the expected image counts before starting training.

Expected:

| Path | Count |
|---|---|
| `gate_data/train/fish` | 800 |
| `gate_data/train/no_fish` | 800 |
| `gate_data/val/fish` | 200 |
| `gate_data/val/no_fish` | 200 |

In [ ]:
import os

base = "gate_data"
total = 0
all_ok = True
expected = {"train/fish": 800, "train/no_fish": 800, "val/fish": 200, "val/no_fish": 200}

print("Gate dataset structure:")
print("-" * 40)
for rel_path, expected_count in expected.items():
    full_path = os.path.join(base, rel_path)
    if os.path.exists(full_path):
        count = len(os.listdir(full_path))
        status = "✓" if count == expected_count else f"✗ (expected {expected_count})"
        if count != expected_count:
            all_ok = False
    else:
        count = 0
        status = "✗ MISSING"
        all_ok = False
    print(f"  {full_path}: {count} images  {status}")
    total += count

print("-" * 40)
print(f"Total: {total} images")
print()
if all_ok:
    print("✓ Dataset looks correct — ready to train.")
else:
    print("✗ Dataset has issues — re-run Cell 4 (gate-download).")

---
## GPU Check — Before Training

Confirm that CUDA is available. If `torch.cuda.is_available()` returns `False`, go to **Runtime → Change Runtime Type → T4 GPU** and re-run from Cell 1.

Training on CPU is possible but will take ~10× longer (~3–4 hours vs ~20–25 min on T4).

In [ ]:
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA available : {cuda_available}")

if cuda_available:
    print(f"GPU device     : {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory     : {mem_gb:.1f} GB")
    print("\n✓ Ready to train on GPU.")
else:
    print("\n⚠ No GPU detected.")
    print("  Go to Runtime → Change Runtime Type → Hardware Accelerator → T4 GPU")
    print("  Then re-run all cells from Cell 1.")

---
## Cell 6 — Train the Gate Classifier

Runs `gate-train` which performs two-phase transfer learning on **MobileNetV3-Small** (pretrained on ImageNet1K):

### Phase 1 — Head Only (5 epochs, lr=1e-3)
All backbone (`model.features`) layers are frozen. Only the new `Linear(in_features, 1)` head is trained. This converges to a reasonable baseline quickly without disrupting ImageNet features.

### Phase 2 — Backbone Fine-tuning (15 epochs, lr=5e-5)
The last 2 blocks of `model.features` are unfrozen. A much lower learning rate fine-tunes these blocks to better recognise fish textures and shapes while keeping earlier feature detectors stable.

### Label convention
`torchvision.datasets.ImageFolder` sorts classes alphabetically:  
`class_to_idx = {'fish': 0, 'no_fish': 1}`  
The training code **flips** labels so `fish → 1.0` (high sigmoid output = fish).
This means in the JS gate worker, `sigmoid(logit) > 0.6` = "fish detected".

### Best weights
The checkpoint with the highest validation accuracy is saved to `runs/gate/fish_gate_best.pt`.

> **Tip:** If the session times out mid-training, see the *Resume After Timeout* cells at the bottom.

In [ ]:
# Train the gate classifier (Phase 1 + Phase 2)
# --phase1-epochs 5   : head-only warm-up
# --phase2-epochs 15  : backbone fine-tuning
# --batch 32          : safe for T4 GPU at 224×224
# --device is auto-detected (cuda if available, cpu otherwise)
!uv run gate-train \
    --phase1-epochs 5 \
    --phase2-epochs 15 \
    --batch 32

---
## Cell 7 — Export to ONNX and Verify

Runs `gate-export` which:

1. Loads `runs/gate/fish_gate_best.pt` (the best checkpoint from training)
2. Exports to `runs/gate/fish_gate.onnx` using `torch.onnx.export` with:
   - **Input name:** `images` (matches the existing disease worker naming)
   - **Input shape:** `float32[1, 3, 224, 224]`
   - **Output name:** `output`
   - **Output shape:** `float32[1, 1]` (raw logit, apply `sigmoid` in JS)
   - **Opset:** 17 (compatible with `onnxruntime-web >= 1.17`)
3. Verifies the exported model using `onnxruntime` (CPU) — runs a random input and checks output shape is `(1, 1)`

### What to do with the ONNX file  
After downloading in Cell 9, place it at:
```
web/public/model/fish_gate.onnx
```
The existing Vite PWA service worker config automatically caches any `*.onnx` under `public/model/` — **no Vite config changes needed**.

In [ ]:
# Export best .pt weights to ONNX and run CPU sanity verification
# Default paths: --weights runs/gate/fish_gate_best.pt
#                --output  runs/gate/fish_gate.onnx
#                --opset   17
!uv run gate-export

---
## Cell 8 — Evaluate the ONNX Model

Runs `gate-evaluate` which loads `fish_gate.onnx` via `onnxruntime` (CPU) and evaluates it on all 400 images in `gate_data/val/` (200 fish + 200 no-fish).

### Acceptance criteria for Phase 1

| Metric | Required |
|---|---|
| Val accuracy | ≥ 90% |
| Precision (fish class) | ≥ 88% |
| Recall (fish class) | ≥ 88% |
| ONNX file size | ≤ 4 MB |

### If criteria are NOT met

| Problem | Fix |
|---|---|
| Accuracy < 90% | Increase `--phase2-epochs` to 20–25 and re-run Cell 6 |
| Low precision | Increase `--threshold` to 0.7 (re-run this cell with `--threshold 0.7`) |
| Low recall | Lower `--threshold` to 0.5 (re-run this cell with `--threshold 0.5`) |
| Size > 4 MB | Should not happen with this model — check that only 1 model was exported |

In [ ]:
# Evaluate the exported ONNX model on the validation split
# --threshold 0.6 : sigmoid probability above which we classify as "fish"
# Adjust threshold and re-run this cell only — no retraining needed
!uv run gate-evaluate --threshold 0.6

---
## Cell 9 — Download the ONNX File

Downloads `fish_gate.onnx` from the Colab session to your local machine.

Also downloads `fish_gate_best.pt` (the PyTorch checkpoint) — save this to Google Drive or your local machine. If the Colab session times out before you can re-export, you can re-upload the `.pt` and jump straight to Cell 7.

### After downloading

Place `fish_gate.onnx` at:
```
fishcareyolo/web/public/model/fish_gate.onnx
```
You are now ready for **Phase 2: Web App Integration** (adding `gate-worker.ts` and `gate-service.ts` to the Vite PWA).

In [ ]:
from google.colab import files

# Download the ONNX model → copy to web/public/model/fish_gate.onnx
print("Downloading fish_gate.onnx ...")
files.download("runs/gate/fish_gate.onnx")

# Download the PyTorch checkpoint → keep as backup in case you need to re-export
print("Downloading fish_gate_best.pt (backup checkpoint) ...")
files.download("runs/gate/fish_gate_best.pt")

print()
print("Next step:")
print("  Copy fish_gate.onnx → fishcareyolo/web/public/model/fish_gate.onnx")
print("  Then start Phase 2 (web app integration).")

---
## Resuming After a Session Timeout

Colab sessions reset after idle time. If your session timed out mid-training:

1. Re-run **Cell 1** (clone repo, checkout `fish-or-notfish`, `cd model/`)
2. Re-run **Cell 2** (`uv sync`)
3. **Upload `fish_gate_best.pt`** via Files panel (if you saved it earlier)
4. Move it to the expected path (see cell below)
5. Skip straight to **Cell 7** (export) — no re-training needed

If you lost the `.pt` checkpoint, restart from **Cell 1** and run all cells again.

In [ ]:
# ── Run this cell ONLY when resuming after a timeout ─────────────────────────
# Assumes you have:
#   1. Re-run Cell 1 (cloned repo, checked out fish-or-notfish, cd model/)
#   2. Re-run Cell 2 (uv sync)
#   3. Uploaded fish_gate_best.pt via the Files panel to /content/fish_gate_best.pt

import os
import shutil

checkpoint_src = "/content/fish_gate_best.pt"
checkpoint_dst = "runs/gate/fish_gate_best.pt"

if not os.path.exists(checkpoint_src):
    print(f"Upload fish_gate_best.pt to Colab first (Files panel → Upload).")
else:
    os.makedirs("runs/gate", exist_ok=True)
    shutil.copy(checkpoint_src, checkpoint_dst)
    print(f"Checkpoint copied to: {checkpoint_dst}")
    print("Now run Cell 7 (gate-export) to re-export the ONNX model.")